In [1]:
!rm -rf /content/document_store
!rm -rf /content/.deepeval
!rm -f /content/lightweight_rag_qa_dataset*.csv
!rm -f /content/rag_qa_dataset_with_demo_rag_outputs.csv

In [2]:
!grep -R "__redwood_timeboxed__" /content 2>/dev/null | head

# RAG Pipeline Evaluation Project

## Part 1: Dataset Preparation and Document Store Creation

Steps 1 to 28 prepare the QA dataset, select test questions, create expected answers, and export source documents into the `document_store` folder.

# ETL part

load EnterpriseRAG-Bench from Hugging Face

In [3]:
# Step 1: Install required library
# "datasets" is the Hugging Face library used to download public datasets.
!pip install datasets -q

## 1. Load EnterpriseRAG-Bench Dataset
This section installs required libraries and loads the documents/questions datasets from Hugging Face.

In [4]:
# Step 2: Import required libraries

# load_dataset is used to load datasets from Hugging Face.
from datasets import load_dataset

# pandas is used to convert the dataset into table format.
import pandas as pd


# Step 3: Load EnterpriseRAG-Bench dataset from Hugging Face

# This dataset has two important parts:
# 1. documents  -> company-like documents used as RAG knowledge base
# 2. questions  -> test questions with expected answers and expected document IDs

documents_dataset = load_dataset(
    "onyx-dot-app/EnterpriseRAG-Bench",
    "documents",
    split="test"
)

questions_dataset = load_dataset(
    "onyx-dot-app/EnterpriseRAG-Bench",
    "questions",
    split="test"
)


# Step 4: Convert Hugging Face dataset into pandas DataFrame
# DataFrame means table format, like Excel/CSV.

documents_df = documents_dataset.to_pandas()
questions_df = questions_dataset.to_pandas()


# Step 5: Check how many rows and columns are present

print("Documents dataset shape:", documents_df.shape)
print("Questions dataset shape:", questions_df.shape)


# Step 6: Show column names
# This helps us understand what fields are available.

print("\nDocuments columns:")
print(documents_df.columns.tolist())

print("\nQuestions columns:")
print(questions_df.columns.tolist())


# Step 7: Preview first few rows

print("\nDocuments preview:")
display(documents_df.head())

print("\nQuestions preview:")
display(questions_df.head())

Documents dataset shape: (511962, 4)
Questions dataset shape: (500, 7)

Documents columns:
['doc_id', 'source_type', 'title', 'content']

Questions columns:
['question_id', 'question_type', 'source_types', 'question', 'expected_doc_ids', 'gold_answer', 'answer_facts']

Documents preview:


,doc_id,source_type,title,content
0,dsid_e54ef48bae78474684a957cf613d47d5,confluence,Runbook: Deploy / Upgrade / Roll Back perf-can...,## Purpose\nThis runbook describes the operati...
1,dsid_229dd48e9b1d466a81ebaffe3ec84469,confluence,Cross-account GPU burst SLO contract and CI li...,Summary\n\nOverview:\nThis document defines th...
2,dsid_aeb0022d62bc43beb6549ba92e5655eb,confluence,Third-Party & Vendor Coordination Playbook for...,Overview\n\nPurpose:\nThis playbook documents ...
3,dsid_926174fc4900408c89c98abde46b7225,confluence,First Production Launch Checklist (Canonical T...,## Purpose\nThis page defines the **canonical ...
4,dsid_d511c6d8daf94f998ce6a3d97462af2d,confluence,Go-Live Runway and Week-3 Stability Playbook,Overview\n\nThis playbook defines the technica...



Questions preview:


,question_id,question_type,source_types,question,expected_doc_ids,gold_answer,answer_facts
0,qst_0001,basic,[github],What are the default size limits for file uplo...,[dsid_ae068ee4aa9640159427cd941bef0238],The default limits are 10 MiB per file (max_fi...,[The default per file upload size limit (max_f...
1,qst_0002,basic,[github],What is the name of the new metric added so SR...,[dsid_9550250a59e74f1bbd5612480b2e7100],The new metric is `stream.timebox_finalized` (...,[The new metric added is named stream.timebox_...
2,qst_0003,basic,[linear],What are the acceptance criteria for the proje...,[dsid_3fd6af404fae48e6b8ea5a57875ef78f],The acceptance criteria are: (1) deliver a sta...,[Deliver a stable design JSON token spec that ...
3,qst_0004,basic,[fireflies],In the meeting about onboarding a SaaS product...,[dsid_6c4c1c875e704f09b4d791d64d7bc7e5],The GCP team said entitlement propagation dela...,[Entitlement propagation delays can occur afte...
4,qst_0005,basic,[gmail],What failover sequence and recovery targets di...,[dsid_8e838ab6a98f4cbcb672d41f210ff89c],MedThink's preferred failover hierarchy is EU ...,[MedThink specified a failover hierarchy of EU...


## 2. Inspect Dataset Structure
This section checks one sample document and one sample question to understand the dataset fields.

In [5]:
# Step 8: Check one full document row
# This helps us understand what one knowledge-base document looks like.

print("One document example:")
display(documents_df.iloc[0])


# Step 9: Check one full question row
# This helps us understand what one test question looks like.

print("One question example:")
display(questions_df.iloc[0])

One document example:


,0
doc_id,dsid_e54ef48bae78474684a957cf613d47d5
source_type,confluence
title,Runbook: Deploy / Upgrade / Roll Back perf-can...
content,## Purpose\nThis runbook describes the operati...


One question example:


,0
question_id,qst_0001
question_type,basic
source_types,[github]
question,What are the default size limits for file uplo...
expected_doc_ids,[dsid_ae068ee4aa9640159427cd941bef0238]
gold_answer,The default limits are 10 MiB per file (max_fi...
answer_facts,[The default per file upload size limit (max_f...


## 3. Select Lightweight QA Test Questions
This section selects 10 basic questions with one clear expected source document.

In [6]:
# Step 10: Select simple/basic questions for our first QA test set

# We are starting with "basic" questions because they are easier to verify.
# Later, we can test complex question types.

basic_questions_df = questions_df[questions_df["question_type"] == "basic"].copy()


# Step 11: Keep only questions that have exactly one expected document
# This makes the first version simple:
# one question -> one correct source document -> one expected answer.

basic_questions_df["expected_doc_count"] = basic_questions_df["expected_doc_ids"].apply(len)

single_doc_questions_df = basic_questions_df[
    basic_questions_df["expected_doc_count"] == 1
].copy()


# Step 12: Select first 10 questions
# We are choosing only 10 because this is the first QA prototype.
# After this works, we can increase the count.

selected_questions_df = single_doc_questions_df.head(10).copy()


# Step 13: Show selected questions

print("Selected questions count:", len(selected_questions_df))

display(
    selected_questions_df[
        [
            "question_id",
            "question_type",
            "question",
            "expected_doc_ids",
            "gold_answer",
            "answer_facts"
        ]
    ]
)

Selected questions count: 10


,question_id,question_type,question,expected_doc_ids,gold_answer,answer_facts
0,qst_0001,basic,What are the default size limits for file uplo...,[dsid_ae068ee4aa9640159427cd941bef0238],The default limits are 10 MiB per file (max_fi...,[The default per file upload size limit (max_f...
1,qst_0002,basic,What is the name of the new metric added so SR...,[dsid_9550250a59e74f1bbd5612480b2e7100],The new metric is `stream.timebox_finalized` (...,[The new metric added is named stream.timebox_...
2,qst_0003,basic,What are the acceptance criteria for the proje...,[dsid_3fd6af404fae48e6b8ea5a57875ef78f],The acceptance criteria are: (1) deliver a sta...,[Deliver a stable design JSON token spec that ...
3,qst_0004,basic,In the meeting about onboarding a SaaS product...,[dsid_6c4c1c875e704f09b4d791d64d7bc7e5],The GCP team said entitlement propagation dela...,[Entitlement propagation delays can occur afte...
4,qst_0005,basic,What failover sequence and recovery targets di...,[dsid_8e838ab6a98f4cbcb672d41f210ff89c],MedThink's preferred failover hierarchy is EU ...,[MedThink specified a failover hierarchy of EU...
5,qst_0006,basic,In the draft spec about extending a routing po...,[dsid_184be937d34a412ab5e61366d54d8ed6],The draft proposes this canonical v1 signal pr...,[The draft spec proposes a canonical v1 priori...
6,qst_0007,basic,In a rolling investigation of a model regressi...,[dsid_72ec4a9962ba43e88acd61abbba1052d],"In the first comparison run, the baseline buil...","[In the first comparison run, the older baseli..."
7,qst_0008,basic,"In the internal shiproom runner notes, what is...",[dsid_5fc2dba9f6ac4af2b49b4f546a4298d0],The rollback plan is considered verified in st...,"[In staging, a verified rollback requires a ti..."
8,qst_0009,basic,"In the EdgePath evaluation email thread, what ...",[dsid_85deb10a652742baaf28af6149600001],Redwood said it wouldn't match the competitor'...,[Redwood did not match the competitors 50 perc...
9,qst_0010,basic,How does the new alerting approach group model...,[dsid_c1a6a71323c04c1ba5445aadea340362],It adds a lightweight token_stage_cohort servi...,[The approach adds a lightweight token_stage_c...


## 4. Create Lightweight QA Dataset
This section maps each selected question to its expected source document and prepares the QA dataset.

In [7]:
# Step 14: Get the matching source document for each selected question

# We create a lookup table using doc_id.
# This helps us quickly find a document by its document ID.

documents_lookup = documents_df.set_index("doc_id")


# Step 15: Create rows for our lightweight QA dataset

lightweight_rows = []

for index, question_row in selected_questions_df.iterrows():

    # Each selected question has exactly one expected document ID.
    expected_doc_id = question_row["expected_doc_ids"][0]

    # Find the matching document from documents_df using that expected_doc_id.
    matching_document = documents_lookup.loc[expected_doc_id]

    # Create one QA test row.
    lightweight_rows.append({
        "test_id": f"RAG_TC_{len(lightweight_rows) + 1:03d}",

        # Unique code created by us for easy tracking.
        "input_code": f"EVL-RAG-{len(lightweight_rows) + 1:03d}",

        # Question details from questions dataset.
        "question_id": question_row["question_id"],
        "question_type": question_row["question_type"],
        "question": question_row["question"],

        # Expected source document details.
        "expected_doc_ids": expected_doc_id,
        "expected_doc_title": matching_document["title"],
        "expected_source_type": matching_document["source_type"],
        "expected_doc_content": str(matching_document["content"]).replace('"__redwood_timeboxed__"', "redwood_timeboxed"),

        # Expected answer details.
        "gold_answer": question_row["gold_answer"],
        "answer_facts": question_row["answer_facts"],

        # These fields will be filled after running the RAG chatbot/model.
        "actual_retrieved_doc_ids": "",
        "actual_output": "",

        # These fields will be filled by our embedding evaluator later.
        "expected_embedding_code": "",
        "actual_embedding_code": "",
        "similarity_score": "",

        # Final QA result fields.
        "faithfulness_check": "Not Checked",
        "hallucination_check": "Not Checked",
        "match_status": "Not Run",
        "remarks": ""
    })


# Step 16: Convert rows into DataFrame

lightweight_rag_df = pd.DataFrame(lightweight_rows)


# Step 17: Preview the lightweight QA dataset

print("Lightweight QA dataset shape:", lightweight_rag_df.shape)

display(lightweight_rag_df.head())

Lightweight QA dataset shape: (10, 20)


,test_id,input_code,question_id,question_type,question,expected_doc_ids,expected_doc_title,expected_source_type,expected_doc_content,gold_answer,answer_facts,actual_retrieved_doc_ids,actual_output,expected_embedding_code,actual_embedding_code,similarity_score,faithfulness_check,hallucination_check,match_status,remarks
0,RAG_TC_001,EVL-RAG-001,qst_0001,basic,What are the default size limits for file uplo...,dsid_ae068ee4aa9640159427cd941bef0238,"add multipart/form-data handling, strict conte...",github,description:\nMotivation: users integrating to...,The default limits are 10 MiB per file (max_fi...,[The default per file upload size limit (max_f...,,,,,,Not Checked,Not Checked,Not Run,
1,RAG_TC_002,EVL-RAG-002,qst_0002,basic,What is the name of the new metric added so SR...,dsid_9550250a59e74f1bbd5612480b2e7100,introduce-server-timebox-and-idempotent-cancel...,github,description:\nContext: production customers ob...,The new metric is `stream.timebox_finalized` (...,[The new metric added is named stream.timebox_...,,,,,,Not Checked,Not Checked,Not Run,
2,RAG_TC_003,EVL-RAG-003,qst_0003,basic,What are the acceptance criteria for the proje...,dsid_3fd6af404fae48e6b8ea5a57875ef78f,Develop interactive tone derivatives and Kappa...,linear,description:\nObjective: Create a deterministi...,The acceptance criteria are: (1) deliver a sta...,[Deliver a stable design JSON token spec that ...,,,,,,Not Checked,Not Checked,Not Run,
3,RAG_TC_004,EVL-RAG-004,qst_0004,basic,In the meeting about onboarding a SaaS product...,dsid_6c4c1c875e704f09b4d791d64d7bc7e5,GCP Marketplace onboarding + billing review (R...,fireflies,summary:\nRedwood and the GCP Marketplace team...,The GCP team said entitlement propagation dela...,[Entitlement propagation delays can occur afte...,,,,,,Not Checked,Not Checked,Not Run,
4,RAG_TC_005,EVL-RAG-005,qst_0005,basic,What failover sequence and recovery targets di...,dsid_8e838ab6a98f4cbcb672d41f210ff89c,Regional fallback priorities & logs — post-cal...,gmail,['From: Rafael Mendes <rafael.mendes@redwoodin...,MedThink's preferred failover hierarchy is EU ...,[MedThink specified a failover hierarchy of EU...,,,,,,Not Checked,Not Checked,Not Run,


In [8]:
# Step 18: Save the lightweight QA dataset as a CSV file

# This CSV is our final ETL output.
# We will use this file for QA testing and embedding comparison later.

lightweight_rag_df.to_csv("lightweight_rag_qa_dataset.csv", index=False)


# Step 19: Confirm the file is saved

print("CSV file created successfully: lightweight_rag_qa_dataset.csv")

CSV file created successfully: lightweight_rag_qa_dataset.csv


Embedding evaluator part

## 5. Generate Expected Embedding Fingerprint Codes
This section converts gold answers into expected embedding fingerprint codes.

In [9]:
# Step 20: Install sentence-transformers
# This library gives us an embedding model.
# Embedding means converting text into numeric meaning code/vector.

!pip install sentence-transformers -q

In [10]:
# Step 21: Load embedding model

# SentenceTransformer is used to convert text into embedding vectors.
from sentence_transformers import SentenceTransformer

# This is a small and commonly used embedding model.
# It converts sentence meaning into a numeric vector.
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

print("Embedding model loaded successfully.")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding model loaded successfully.


In [11]:
# Step 22: Create expected embedding fingerprint code from gold_answer

# hashlib is used to create a small unique fingerprint from the full embedding.
# This makes the code short and readable, while still being based on the full embedding.
import hashlib
import numpy as np


def create_embedding_fingerprint(text, prefix="EXP"):
    # If text is empty or missing, return empty code.
    if pd.isna(text) or str(text).strip() == "":
        return ""

    # Convert the full text meaning into an embedding vector.
    # This vector represents the meaning of the expected answer.
    embedding_vector = embedding_model.encode(str(text))

    # Round the full embedding vector slightly before hashing.
    # This keeps the fingerprint stable and avoids tiny decimal noise.
    rounded_vector = np.round(embedding_vector, 6)

    # Convert the full rounded vector into bytes.
    # Hashing needs byte format, so we convert the numeric vector into bytes.
    vector_bytes = rounded_vector.tobytes()

    # Create a short hash/fingerprint from the full embedding vector.
    # We use only the first 8 characters to keep the code simple.
    fingerprint = hashlib.sha256(vector_bytes).hexdigest()[:8]

    # Return a small readable embedding code.
    # Example: EXP-a91f23c8
    return f"{prefix}-{fingerprint}"


# Apply the embedding fingerprint generator to gold_answer.
# This creates a short expected embedding code based on the full expected answer meaning.
lightweight_rag_df["expected_embedding_code"] = lightweight_rag_df["gold_answer"].apply(
    lambda text: create_embedding_fingerprint(text, prefix="EXP")
)


# Show the generated expected embedding codes.
display(
    lightweight_rag_df[
        ["test_id", "gold_answer", "expected_embedding_code"]
    ].head()
)


,test_id,gold_answer,expected_embedding_code
0,RAG_TC_001,The default limits are 10 MiB per file (max_fi...,EXP-0b583284
1,RAG_TC_002,The new metric is `stream.timebox_finalized` (...,EXP-e07dadfc
2,RAG_TC_003,The acceptance criteria are: (1) deliver a sta...,EXP-f4193af0
3,RAG_TC_004,The GCP team said entitlement propagation dela...,EXP-4fe10deb
4,RAG_TC_005,MedThink's preferred failover hierarchy is EU ...,EXP-77f7105f


In [12]:
# Step 23: Save updated CSV with expected embedding codes

lightweight_rag_df.to_csv("lightweight_rag_qa_dataset_with_expected_codes.csv", index=False)

print("Updated CSV saved successfully: lightweight_rag_qa_dataset_with_expected_codes.csv")

Updated CSV saved successfully: lightweight_rag_qa_dataset_with_expected_codes.csv


 Developer/QA note:
 Before running the comparison step, paste or load the RAG model outputs into the actual_output column.
 actual_output should contain the answer returned by the RAG chatbot/model for each question.

## 6. Prepare for Actual Output Comparison
This section loads the prepared CSV and checks whether actual model outputs are available.

In [13]:
from google.colab import files

uploaded = files.upload()

Saving lightweight_rag_qa_dataset_with_expected_codes (14).csv to lightweight_rag_qa_dataset_with_expected_codes (14).csv


In [14]:
import pandas as pd

qa_df = pd.read_csv("lightweight_rag_qa_dataset_with_expected_codes.csv")

print("CSV loaded successfully")
print(qa_df.shape)

CSV loaded successfully
(10, 20)


In [15]:

# Step 24: Load the updated CSV for actual-output comparison stage

# This CSV already contains:
# - selected RAG test questions
# - expected source documents
# - gold_answer
# - expected_embedding_code
# - empty fields for actual_output and comparison results

import pandas as pd

qa_df = pd.read_csv("lightweight_rag_qa_dataset_with_expected_codes.csv")


# Step 25: Check current QA dataset status

# Total number of test cases available in the CSV.
total_test_cases = len(qa_df)

# Count rows where actual_output is empty.
# actual_output must be filled only after running the RAG model/chatbot.
empty_actual_output_count = qa_df["actual_output"].isna().sum() + (
    qa_df["actual_output"].astype(str).str.strip() == ""
).sum()

# Count rows where expected_embedding_code is already created.
expected_code_count = (
    qa_df["expected_embedding_code"].notna()
    & (qa_df["expected_embedding_code"].astype(str).str.strip() != "")
).sum()


print("Total test cases:", total_test_cases)
print("Rows with expected embedding code:", expected_code_count)
print("Rows waiting for actual model output:", empty_actual_output_count)


# Step 26: Display important QA tracking columns

# This preview helps us verify that:
# - expected answer code is already available
# - actual_output is still waiting for model response
# - result columns are ready for the next comparison phase

display(
    qa_df[
        [
            "test_id",
            "question",
            "gold_answer",
            "expected_embedding_code",
            "actual_output",
            "actual_embedding_code",
            "similarity_score",
            "match_status"
        ]
    ].head()
)


Total test cases: 10
Rows with expected embedding code: 10
Rows waiting for actual model output: 10


,test_id,question,gold_answer,expected_embedding_code,actual_output,actual_embedding_code,similarity_score,match_status
0,RAG_TC_001,What are the default size limits for file uplo...,The default limits are 10 MiB per file (max_fi...,EXP-0b583284,NaN,NaN,NaN,Not Run
1,RAG_TC_002,What is the name of the new metric added so SR...,The new metric is `stream.timebox_finalized` (...,EXP-e07dadfc,NaN,NaN,NaN,Not Run
2,RAG_TC_003,What are the acceptance criteria for the proje...,The acceptance criteria are: (1) deliver a sta...,EXP-f4193af0,NaN,NaN,NaN,Not Run
3,RAG_TC_004,In the meeting about onboarding a SaaS product...,The GCP team said entitlement propagation dela...,EXP-4fe10deb,NaN,NaN,NaN,Not Run
4,RAG_TC_005,What failover sequence and recovery targets di...,MedThink's preferred failover hierarchy is EU ...,EXP-77f7105f,NaN,NaN,NaN,Not Run


In [16]:
# Convert these columns into text type before storing text values.
qa_df["actual_output"] = qa_df["actual_output"].astype("object")
qa_df["actual_embedding_code"] = qa_df["actual_embedding_code"].astype("object")

In [17]:
qa_df = qa_df.head(2).copy()

## Demo RAG Pipeline Implementation Plan

This section creates a simple demo RAG pipeline only for testing the QA evaluation workflow.

Implementation flow:

1. Create a `document_store` folder.
2. Export each expected document from the CSV into a separate `.txt` file inside `document_store`.
3. Add the document file path back into the dataset as `expected_doc_path`.
4. Add correct documents and later add distractor/wrong documents into the same `document_store`.
5. Demo RAG will read documents from the `document_store` folder.
6. Retrieval will use hybrid search:
   - keyword matching score
   - embedding similarity score
7. The best matching document will be selected as `actual_retrieved_doc_ids`.
8. The answer will be created by extracting the most relevant sentences from the selected document.
9. The generated answer will be stored as `actual_output`.
10. Existing QA checks will run:
    - retrieval match
    - semantic similarity
    - fact-level meaning check
    - overall QA status

Note:

This is not a production RAG chatbot. It is a QA test setup created to generate actual outputs and validate the RAG evaluation workflow end-to-end.

In [18]:
# Step 27: Create document_store folder and export expected documents as .txt files

# This step separates document content from the CSV.
# CSV will keep QA test case details.
# document_store folder will keep source documents like a real RAG knowledge base.

import os
import re

# Create document_store folder if it does not already exist
document_store_path = "document_store"
os.makedirs(document_store_path, exist_ok=True)


def clean_filename(text):
    """
    Create a safe file name from document id/title.
    This removes characters that are not safe for file names.
    """
    text = str(text)
    text = re.sub(r"[^a-zA-Z0-9_-]", "_", text)
    return text[:120]


# Create a new column to store each exported document file path
qa_df["expected_doc_path"] = ""

# Export each expected document content into a separate text file
for index, row in qa_df.iterrows():
    test_id = row["test_id"]
    doc_id = row["expected_doc_ids"]
    doc_title = row["expected_doc_title"]
    doc_content = row["expected_doc_content"]

    # Create readable and unique file name
    file_name = f"{test_id}_{clean_filename(doc_id)}.txt"
    file_path = os.path.join(document_store_path, file_name)

    # Write document content into .txt file
    with open(file_path, "w", encoding="utf-8") as file:
        file.write(f"doc_id: {doc_id}\n")
        file.write(f"title: {doc_title}\n\n")
        file.write(str(doc_content))

    # Store file path back into dataframe
    qa_df.loc[index, "expected_doc_path"] = file_path


print("Document store created successfully.")
print("Total documents exported:", len(qa_df))
print("Folder name:", document_store_path)

display(
    qa_df[
        [
            "test_id",
            "expected_doc_ids",
            "expected_doc_title",
            "expected_doc_path"
        ]
    ]
)

Document store created successfully.
Total documents exported: 2
Folder name: document_store


,test_id,expected_doc_ids,expected_doc_title,expected_doc_path
0,RAG_TC_001,dsid_ae068ee4aa9640159427cd941bef0238,"add multipart/form-data handling, strict conte...",document_store/RAG_TC_001_dsid_ae068ee4aa96401...
1,RAG_TC_002,dsid_9550250a59e74f1bbd5612480b2e7100,introduce-server-timebox-and-idempotent-cancel...,document_store/RAG_TC_002_dsid_9550250a59e74f1...


In [19]:
# Step 28: Check exported documents inside document_store folder

# This confirms that our document files are created properly.
# The demo RAG system will read documents from this folder.

exported_files = os.listdir(document_store_path)

print("Total files in document_store:", len(exported_files))
print("First few files:")

for file_name in exported_files[:5]:
    print(file_name)

Total files in document_store: 2
First few files:
RAG_TC_001_dsid_ae068ee4aa9640159427cd941bef0238.txt
RAG_TC_002_dsid_9550250a59e74f1bbd5612480b2e7100.txt


## Part 2: Demo RAG Runtime Pipeline

From Step 29 onward, the notebook reads documents from `document_store`, creates embeddings, retrieves top-k documents, generates answers, and runs evaluation checks.

In [20]:
# Step 29: Load documents from document_store folder

# This step reads the exported .txt files.
# Now the demo RAG system will use file-based documents instead of reading document content directly from CSV.

document_store = []

for file_name in os.listdir(document_store_path):
    if file_name.endswith(".txt"):
        file_path = os.path.join(document_store_path, file_name)

        with open(file_path, "r", encoding="utf-8") as file:
            content = file.read()

        document_store.append(
            {
                "file_name": file_name,
                "file_path": file_path,
                "content": content
            }
        )

print("Documents loaded from document_store:", len(document_store))

# Show one loaded document sample
print("Sample file:", document_store[0]["file_name"])
print("Sample content preview:")
print(document_store[0]["content"][:500])

Documents loaded from document_store: 2
Sample file: RAG_TC_001_dsid_ae068ee4aa9640159427cd941bef0238.txt
Sample content preview:
doc_id: dsid_ae068ee4aa9640159427cd941bef0238
title: add multipart/form-data handling, strict content-type validation, and payload limit enforcement for API tool/file inputs

description:
Motivation: users integrating tool/function calling and file uploads via the OpenAI-compatibility endpoints were sending a wide variety of content-types and very large multipart requests which caused inconsistent runtime behavior, high memory pressure, and unexpected acceptance of unsupported payloads. This PR 


In [21]:
"""
# Step 29A: Create document embeddings once for vector retrieval

# This follows proper RAG architecture.
# Documents are loaded from document_store folder.
# Then all documents are converted into embeddings one time.
# During retrieval, we will reuse these embeddings.

document_texts = [document["content"] for document in document_store]

document_embeddings = embedding_model.encode(
    document_texts,
    convert_to_tensor=True
)

print("Document embeddings created successfully.")
print("Total documents embedded:", len(document_texts))
print("Total embedding vectors:", len(document_embeddings))
"""

'\n# Step 29A: Create document embeddings once for vector retrieval\n\n# This follows proper RAG architecture.\n# Documents are loaded from document_store folder.\n# Then all documents are converted into embeddings one time.\n# During retrieval, we will reuse these embeddings.\n\ndocument_texts = [document["content"] for document in document_store]\n\ndocument_embeddings = embedding_model.encode(\n    document_texts,\n    convert_to_tensor=True\n)\n\nprint("Document embeddings created successfully.")\nprint("Total documents embedded:", len(document_texts))\nprint("Total embedding vectors:", len(document_embeddings))\n'

In [22]:
"""
# Step 30: Create vector retrieval logic

# This step retrieves the top-k most relevant documents for a question.
#
# Flow:
# 1. User question is converted into an embedding.
# 2. The question embedding is compared with all stored document embeddings.
# 3. Documents are ranked by similarity score.
# 4. Top-k documents are returned in ranked order.

from sentence_transformers import util


def vector_retrieve(question, document_store, document_embeddings, top_k=3):
    """""""
    Retrieves top-k relevant documents for the given question.

    Input:
    - question: user question
    - document_store: documents loaded from document_store folder
    - document_embeddings: embeddings created once for all documents
    - top_k: number of documents to retrieve

    Output:
    - best_document: rank 1 document
    - top_k_results: top-k documents in ranked order
    """"""

    # The user question is normal text.
    # To compare it with documents, we first convert the question into vector format.
    # This vector represents the meaning of the question.
    question_embedding = embedding_model.encode(
        question,
        convert_to_tensor=True
    )

    # Now compare the question vector with all document vectors.
    # This gives one similarity score for each document.
    # Higher score means that document is more related to the question.
    similarity_scores = util.cos_sim(
        question_embedding,
        document_embeddings
    )[0]

    retrieval_results = []

    # Store each document with its similarity score.
    # This makes the ranking visible for QA checking.
    # In some vector databases, this ranking happens internally.
    # Here we store it manually so we can inspect rank, file name, and score.
    for index, document in enumerate(document_store):
        retrieval_results.append(
            {
                "rank": index + 1,
                "file_name": document["file_name"],
                "file_path": document["file_path"],
                "content": document["content"],
                "similarity_score": round(float(similarity_scores[index]), 4)
            }
        )

    # Sort documents by highest similarity score.
    # Rank 1 should be the most relevant document.
    retrieval_results = sorted(
        retrieval_results,
        key=lambda item: item["similarity_score"],
        reverse=True
    )

    # Update rank after sorting.
    # After sorting, rank 1 means best matching document.
    for rank_index, item in enumerate(retrieval_results):
        item["rank"] = rank_index + 1

    # Keep only top-k documents.
    # Example: if top_k=3, keep only top 3 retrieved documents.
    top_k_results = retrieval_results[:top_k]

    # First document is the best retrieved document.
    best_document = top_k_results[0]

    return best_document, top_k_results


# Test vector retrieval for first question.
sample_question = qa_df.loc[0, "question"]

# Run retrieval for one sample question.
# best_document = rank 1 document.
# top_k_results = top 3 retrieved documents.
best_document, top_k_results = vector_retrieve(
    question=sample_question,
    document_store=document_store,
    document_embeddings=document_embeddings,
    top_k=3
)

print("Question:", sample_question)
print("Best retrieved document:", best_document["file_name"])
print("Top-k retrieved documents:")

for result in top_k_results:
    print(
        "Rank:", result["rank"],
        "| File:", result["file_name"],
        "| Score:", result["similarity_score"]
    )

"""

'\n# Step 30: Create vector retrieval logic\n\n# This step retrieves the top-k most relevant documents for a question.\n#\n# Flow:\n# 1. User question is converted into an embedding.\n# 2. The question embedding is compared with all stored document embeddings.\n# 3. Documents are ranked by similarity score.\n# 4. Top-k documents are returned in ranked order.\n\nfrom sentence_transformers import util\n\n\ndef vector_retrieve(question, document_store, document_embeddings, top_k=3):\n    "\n    Retrieves top-k relevant documents for the given question.\n\n    Input:\n    - question: user question\n    - document_store: documents loaded from document_store folder\n    - document_embeddings: embeddings created once for all documents\n    - top_k: number of documents to retrieve\n\n    Output:\n    - best_document: rank 1 document\n    - top_k_results: top-k documents in ranked order\n    \n\n    # The user question is normal text.\n    # To compare it with documents, we first convert the qu

In [23]:
# Step 30A: Create LangChain vector store retrieval logic

# This step replaces the old manual retrieval logic with professional vector-store retrieval.
#
# Flow:
# 1. Convert loaded documents into LangChain Document objects.
# 2. Store those documents inside an in-memory vector store.
# 3. Use similarity_search_with_score() to retrieve top-k relevant documents.
#
# This follows the same RAG retrieval idea:
# question -> vector store -> top-k relevant documents


!pip install langchain-core -q

from langchain_core.documents import Document
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_core.embeddings import Embeddings


class SentenceTransformerEmbeddings(Embeddings):
    """
    This wrapper lets LangChain use our existing SentenceTransformer embedding model.
    """

    def __init__(self, embedding_model):
        self.embedding_model = embedding_model

    def embed_documents(self, texts):
        """
        Converts multiple document texts into embeddings.
        """
        embeddings = self.embedding_model.encode(list(texts))
        return embeddings.tolist()

    def embed_query(self, text):
        """
        Converts one user question into an embedding.
        """
        embedding = self.embedding_model.encode(str(text))
        return embedding.tolist()


def extract_doc_id_from_file_content(content):
    """
    Extracts doc_id from the first line of exported document file.
    """
    first_line = str(content).split("\n")[0]

    if first_line.startswith("doc_id:"):
        return first_line.replace("doc_id:", "").strip()

    return ""


# Convert loaded documents into LangChain Document objects.
langchain_documents = []

for document in document_store:
    doc_id = extract_doc_id_from_file_content(document["content"])

    langchain_documents.append(
        Document(
            page_content=document["content"],
            metadata={
                "doc_id": doc_id,
                "file_name": document["file_name"],
                "file_path": document["file_path"]
            }
        )
    )


# Create LangChain embedding wrapper using existing embedding_model.
langchain_embedding_model = SentenceTransformerEmbeddings(embedding_model)


# Create in-memory vector store.
vector_store = InMemoryVectorStore(
    embedding=langchain_embedding_model
)


# Add documents into vector store.
vector_store.add_documents(langchain_documents)


def vector_retrieve(question, document_store=None, document_embeddings=None, top_k=2):
    """
    Retrieves top-k relevant documents using LangChain vector store.

    Input:
    - question: user question
    - top_k: number of documents to retrieve

    Output:
    - best_document: rank 1 document
    - top_k_results: top-k documents in ranked order
    """

    search_results = vector_store.similarity_search_with_score(
        query=question,
        k=top_k
    )

    retrieval_results = []

    for rank_index, result in enumerate(search_results):
        retrieved_document = result[0]
        similarity_score = result[1]

        retrieval_results.append(
            {
                "rank": rank_index + 1,
                "file_name": retrieved_document.metadata["file_name"],
                "file_path": retrieved_document.metadata["file_path"],
                "content": retrieved_document.page_content,
                "similarity_score": round(float(similarity_score), 4)
            }
        )

    best_document = retrieval_results[0]

    return best_document, retrieval_results


# Test LangChain vector store retrieval for first question.
sample_question = qa_df.loc[0, "question"]

best_document, top_k_results = vector_retrieve(
    question=sample_question,
    top_k=2
)

print("Question:", sample_question)
print("Best retrieved document:", best_document["file_name"])
print("Top-k retrieved documents:")

for result in top_k_results:
    print(
        "Rank:", result["rank"],
        "| File:", result["file_name"],
        "| Score:", result["similarity_score"]
    )

Question: What are the default size limits for file uploads and total request size for the new multipart upload support on the OpenAI-compatible API endpoints?
Best retrieved document: RAG_TC_001_dsid_ae068ee4aa9640159427cd941bef0238.txt
Top-k retrieved documents:
Rank: 1 | File: RAG_TC_001_dsid_ae068ee4aa9640159427cd941bef0238.txt | Score: 0.7318
Rank: 2 | File: RAG_TC_002_dsid_9550250a59e74f1bbd5612480b2e7100.txt | Score: 0.3154


### Manual Retrieval vs LangChain Vector Store Retrieval

| Area | Old Step 29A + Step 30 Manual Way | New Step 30A LangChain Way |
| --- | --- | --- |
| Document format | Python dictionary | LangChain `Document` object |
| Embedding creation | Manually calls `embedding_model.encode()` | Vector store calls embedding through wrapper |
| Embedding storage | Stored in `document_embeddings` variable | Stored inside `InMemoryVectorStore` |
| Question embedding | Manually creates question embedding | Vector store creates it internally |
| Similarity comparison | Manually uses `util.cos_sim()` | Vector store handles similarity search |
| Ranking | Manually sorts by score | Vector store returns ranked results |
| Top-k retrieval | Manually takes first 3 documents | Uses `similarity_search_with_score(query, k=3)` |
| Code style | Custom learning implementation | Professional RAG-style implementation |
| Output | `best_document`, `top_k_results` | Same output maintained |
| Impact on Step 31/32 | Works, but manual backend | Same flow, cleaner backend |

### Model Usage in This RAG QA Evaluation Notebook

This notebook uses two different models for two different purposes.

1. **Local Hugging Face model for answer generation**

   The local Hugging Face model is used to generate the demo RAG `actual_output`.

   This is done in:

   ```text
   Step 31: Generate clean demo RAG response
   Method: extract_relevant_answer()
   ```

   Flow:

   ```text
   Question + Top-k retrieved documents
   → Local Hugging Face model
   → Generated answer / actual_output
   ```

   In this notebook, “local model” means the model is downloaded from Hugging Face into the Colab runtime and runs inside Colab.

2. **Groq model with DeepEval for evaluation**

   Groq is used as the evaluation judge through DeepEval.

   This is done in:

   ```text
   DeepEval Contextual Precision section
   Method: ContextualPrecisionMetric()
   ```

   Flow:

   ```text
   Question + Gold answer + Top-k retrieved documents
   → DeepEval Contextual Precision metric using Groq
   → Evaluation score and reason
   ```

Simple meaning:

```text
Hugging Face model = generates the demo RAG answer
Groq + DeepEval = evaluates the RAG retrieval/context quality
```

In [24]:
# Step 31: Generate clean demo RAG response from retrieved document

# This step creates a clean actual_output from the retrieved document.
#
# Existing flow is kept:
# question -> retrieve best document -> generate clean response from that document
#
# Output format:
# Answer: <short direct answer>
# Source: <retrieved document id>

!pip install transformers torch -q

from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch


def extract_doc_id_from_file_content(content):
    """
    Extracts doc_id from the retrieved document content.
    """
    first_line = str(content).split("\n")[0]

    if first_line.startswith("doc_id:"):
        return first_line.replace("doc_id:", "").strip()

    return ""


def clean_document_text_for_answer_generation(text):
    """
    Creates a cleaned temporary copy of retrieved document text.
    Original document content is not changed.
    """
    lines = str(text).splitlines()

    cleaned_lines = []

    for line in lines:
        line = line.strip()

        if line.startswith("doc_id:"):
            continue

        if line.startswith("title:"):
            continue

        if line == "":
            continue

        cleaned_lines.append(line)

    return " ".join(cleaned_lines)


# Load local instruction/text generation model.
# This model generates a clean answer from question + retrieved document.
#rag_response_model_name = "google/flan-t5-base"
rag_response_model_name = "google/flan-t5-large"

rag_tokenizer = AutoTokenizer.from_pretrained(rag_response_model_name)
rag_response_model = AutoModelForSeq2SeqLM.from_pretrained(rag_response_model_name)



def extract_relevant_answer(question, retrieved_contexts, top_n=3):
    """
    Generates a clean demo RAG response from the retrieved document.

    Input:
    - question: user question
    - document_text: retrieved document content
    - top_n: kept only for Step 32 compatibility

    Output:
    - clean answer with source document id
    """

    cleaned_contexts = [
        clean_document_text_for_answer_generation(context)
        for context in retrieved_contexts
    ]

    context = "\n\n".join(cleaned_contexts)[:2500]

    prompt = f"""
    You are a RAG assistant.

    Use the context to answer the question.
    Write the answer in 1 to 3 complete sentences.
    Include all important numbers, names, steps, or conditions needed to answer the question.
    Do not answer with only one word or a short phrase.
    Do not include unrelated context.

    Question:
    {question}

    Context:
    {context}

    Final Answer:
    """

    inputs = rag_tokenizer(
        prompt,
        return_tensors="pt",
        max_length=1024,
        truncation=True
    )

    with torch.no_grad():
        outputs = rag_response_model.generate(
            **inputs,
            max_new_tokens=120,
            num_beams=4,
            early_stopping=True
        )

    answer = rag_tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    ).strip()

    if answer == "":
        answer = "Answer not found in retrieved document."

    return f"Answer: {answer}"


best_document, top_k_results = vector_retrieve(
    question=sample_question,
    top_k=2
)

sample_top_k_contexts = [
    document["content"]
    for document in top_k_results
]

demo_answer = extract_relevant_answer(
    sample_question,
    sample_top_k_contexts,
    top_n=3
)

print("Generated demo answer:")
print(demo_answer)

Loading weights:   0%|          | 0/558 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


Generated demo answer:
Answer: 10MiB, 50MiB, 20.


In [25]:
print("qa_df exists:", "qa_df" in globals())

if "qa_df" in globals():
    print(qa_df.shape)
    print(qa_df.columns.tolist())

qa_df exists: True
(2, 21)
['test_id', 'input_code', 'question_id', 'question_type', 'question', 'expected_doc_ids', 'expected_doc_title', 'expected_source_type', 'expected_doc_content', 'gold_answer', 'answer_facts', 'actual_retrieved_doc_ids', 'actual_output', 'expected_embedding_code', 'actual_embedding_code', 'similarity_score', 'faithfulness_check', 'hallucination_check', 'match_status', 'remarks', 'expected_doc_path']


In [26]:
import pandas as pd

qa_df = pd.read_csv("lightweight_rag_qa_dataset_with_expected_codes.csv")

qa_df["actual_output"] = qa_df["actual_output"].astype("object")
qa_df["actual_embedding_code"] = qa_df["actual_embedding_code"].astype("object")

print("qa_df loaded:", qa_df.shape)

qa_df loaded: (10, 20)


In [27]:
# Step 32: Generate actual output using traced RAG pipeline

# This step runs the demo RAG pipeline for every question.
#
# Final flow:
# 1. Loop through every question in qa_df.
# 2. Use vector_retrieve() to retrieve top-k relevant documents.
# 3. Send original question + top-k contents to extract_relevant_answer().
# 4. Store actual_output in CSV for final report.
# 5. Store retrieved contexts inside DeepEval trace for professional evaluation style.

!pip install deepeval -q

from deepeval.tracing import observe, update_current_trace


@observe(name="rag_pipeline_run")
def run_traced_rag_pipeline(question, expected_output=None, top_k=2):
    """
    Runs one complete RAG flow and records important data into DeepEval trace.

    This function does:
    - retrieves top-k documents
    - generates answer using Step 31 function
    - stores actual output and retrieved contexts into DeepEval trace

    Input:
    - question: user question
    - expected_output: gold_answer used by DeepEval metrics
    - top_k: number of documents to retrieve

    Output:
    - generated_answer: answer generated by demo RAG model
    """

    # Step 1: Retrieve top-k relevant documents using LangChain vector store.
    best_document, top_k_results = vector_retrieve(
        question=question,
        top_k=top_k
    )

    # Step 2: Extract retrieved document contents in ranked order.
    top_k_contexts = [
        document["content"]
        for document in top_k_results
    ]

    # Step 3: Generate answer using Step 31 answer-generation function.
    generated_answer = extract_relevant_answer(
        question,
        top_k_contexts,
        top_n=top_k
    )

    # Step 4: Store RAG execution details inside DeepEval trace.
    # Full retrieved contexts are kept in trace, not in CSV.
    update_current_trace(
        output=generated_answer,
        expected_output=expected_output,
        retrieval_context=top_k_contexts
    )

    return generated_answer


# Create only final answer column.
qa_df["actual_output"] = ""


# Run traced RAG pipeline for every test case.
for index, row in qa_df.iterrows():

    generated_answer = run_traced_rag_pipeline(
        question=row["question"],
        expected_output=row["gold_answer"],
        top_k=2
    )

    # Store only final answer in CSV.
    # Retrieved contexts are stored in DeepEval trace.
    qa_df.loc[index, "actual_output"] = generated_answer


# Display updated output.
display(
    qa_df[
        [
            "test_id",
            "question",
            "expected_doc_ids",
            "actual_output"
        ]
    ]
)

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
huggingface-hub 1.23.0 requires click<9.0.0,>=8.4.2, but you have click 8.3.3 which is incompatible.


[Confident AI Trace Log]  No Confident AI API key found. Skipping trace posting.

[Confident AI Trace Log]  No Confident AI API key found. Skipping trace posting.

[Confident AI Trace Log]  No Confident AI API key found. Skipping trace posting.

[Confident AI Trace Log]  No Confident AI API key found. Skipping trace posting.

...

[Confident AI Trace Log]  No Confident AI API key found. Skipping trace posting.

[Confident AI Trace Log]  No Confident AI API key found. Skipping trace posting.

[Confident AI Trace Log]  No Confident AI API key found. Skipping trace posting.

[Confident AI Trace Log]  No Confident AI API key found. Skipping trace posting.

,test_id,question,expected_doc_ids,actual_output
0,RAG_TC_001,What are the default size limits for file uplo...,dsid_ae068ee4aa9640159427cd941bef0238,"Answer: 10MiB, 50MiB, 20."
1,RAG_TC_002,What is the name of the new metric added so SR...,dsid_9550250a59e74f1bbd5612480b2e7100,Answer: timeboxed
2,RAG_TC_003,What are the acceptance criteria for the proje...,dsid_3fd6af404fae48e6b8ea5a57875ef78f,Answer: This PR implements a server-side strea...
3,RAG_TC_004,In the meeting about onboarding a SaaS product...,dsid_6c4c1c875e704f09b4d791d64d7bc7e5,Answer: A predictable server-side safety net t...
4,RAG_TC_005,What failover sequence and recovery targets di...,dsid_8e838ab6a98f4cbcb672d41f210ff89c,Answer: This PR implements a server-side strea...
5,RAG_TC_006,In the draft spec about extending a routing po...,dsid_184be937d34a412ab5e61366d54d8ed6,Answer: This PR implements a server-side strea...
6,RAG_TC_007,In a rolling investigation of a model regressi...,dsid_72ec4a9962ba43e88acd61abbba1052d,Answer: 3.0.
7,RAG_TC_008,"In the internal shiproom runner notes, what is...",dsid_5fc2dba9f6ac4af2b49b4f546a4298d0,Answer: Timeboxed=true.
8,RAG_TC_009,"In the EdgePath evaluation email thread, what ...",dsid_85deb10a652742baaf28af6149600001,Answer: Redwood_timeboxed.
9,RAG_TC_010,How does the new alerting approach group model...,dsid_c1a6a71323c04c1ba5445aadea340362,Answer: The new alerting approach group model-...


In [28]:
# Step 33: Save dataset with demo RAG generated outputs

# This CSV now contains:
# - expected answers
# - expected document IDs
# - actual retrieved document IDs from demo RAG
# - actual outputs from demo RAG
# - actual embedding codes

demo_rag_output_file = "rag_qa_dataset_with_demo_rag_outputs.csv"

qa_df.to_csv(demo_rag_output_file, index=False)

print("Demo RAG output CSV saved successfully:", demo_rag_output_file)

Demo RAG output CSV saved successfully: rag_qa_dataset_with_demo_rag_outputs.csv


## 8. Retrieval Document Check
This section checks whether the RAG system retrieved the expected source document.

In [29]:
# Step 28: Retrieval document check

# This logic checks whether the RAG system retrieved the correct source document.
# It works for all rows in the dataset.
# If actual_retrieved_doc_ids is empty, the row is marked as Not Run.

# Create result columns for retrieval evaluation.
qa_df["retrieval_match_status"] = ""
qa_df["retrieval_remarks"] = ""


def normalize_doc_ids(value):
    # Convert document ID values into a clean list.
    # This handles:
    # - empty values
    # - single document ID as text
    # - list-like document IDs saved as text

    if pd.isna(value) or str(value).strip() == "":
        return []

    value = str(value).strip()

    # Remove brackets and quotes if IDs are stored like ['doc_id']
    value = value.replace("[", "").replace("]", "").replace("'", "").replace('"', "")

    # Split by comma in case multiple document IDs are present.
    doc_ids = [item.strip() for item in value.split(",") if item.strip()]

    return doc_ids


# Run retrieval check for every row.
for row_index, row in qa_df.iterrows():

    expected_doc_ids = normalize_doc_ids(row["expected_doc_ids"])
    actual_doc_ids = normalize_doc_ids(row["actual_retrieved_doc_ids"])

    # If actual retrieved document is not available, we cannot run retrieval check yet.
    if len(actual_doc_ids) == 0:
        qa_df.loc[row_index, "retrieval_match_status"] = "Not Run"
        qa_df.loc[row_index, "retrieval_remarks"] = "Actual retrieved document ID is not available yet."
        continue

    # Check whether any expected document ID is present in actual retrieved document IDs.
    is_match = any(doc_id in actual_doc_ids for doc_id in expected_doc_ids)

    if is_match:
        qa_df.loc[row_index, "retrieval_match_status"] = "Pass"
        qa_df.loc[row_index, "retrieval_remarks"] = "Expected document was retrieved."
    else:
        qa_df.loc[row_index, "retrieval_match_status"] = "Fail"
        qa_df.loc[row_index, "retrieval_remarks"] = "Expected document was not retrieved."


# Show retrieval check result.
display(
    qa_df[
        [
            "test_id",
            "expected_doc_ids",
            "actual_retrieved_doc_ids",
            "retrieval_match_status",
            "retrieval_remarks"
        ]
    ]
)

,test_id,expected_doc_ids,actual_retrieved_doc_ids,retrieval_match_status,retrieval_remarks
0,RAG_TC_001,dsid_ae068ee4aa9640159427cd941bef0238,NaN,Not Run,Actual retrieved document ID is not available ...
1,RAG_TC_002,dsid_9550250a59e74f1bbd5612480b2e7100,NaN,Not Run,Actual retrieved document ID is not available ...
2,RAG_TC_003,dsid_3fd6af404fae48e6b8ea5a57875ef78f,NaN,Not Run,Actual retrieved document ID is not available ...
3,RAG_TC_004,dsid_6c4c1c875e704f09b4d791d64d7bc7e5,NaN,Not Run,Actual retrieved document ID is not available ...
4,RAG_TC_005,dsid_8e838ab6a98f4cbcb672d41f210ff89c,NaN,Not Run,Actual retrieved document ID is not available ...
5,RAG_TC_006,dsid_184be937d34a412ab5e61366d54d8ed6,NaN,Not Run,Actual retrieved document ID is not available ...
6,RAG_TC_007,dsid_72ec4a9962ba43e88acd61abbba1052d,NaN,Not Run,Actual retrieved document ID is not available ...
7,RAG_TC_008,dsid_5fc2dba9f6ac4af2b49b4f546a4298d0,NaN,Not Run,Actual retrieved document ID is not available ...
8,RAG_TC_009,dsid_85deb10a652742baaf28af6149600001,NaN,Not Run,Actual retrieved document ID is not available ...
9,RAG_TC_010,dsid_c1a6a71323c04c1ba5445aadea340362,NaN,Not Run,Actual retrieved document ID is not available ...


In [30]:
"""

## 9. Context Precision Check

# This section checks whether the retrieved document/context is useful
# for answering the question.
#
# Simple meaning:
# - Retrieval check only checks whether the expected document ID was retrieved.
# - Context Precision checks whether the retrieved document content is useful.
#
# Correct comparison:
# gold_answer  vs  retrieved document content


def get_retrieved_document_content(retrieved_doc_id, document_store):
    ''''
    Finds the retrieved document content using actual_retrieved_doc_ids.

    Here document_store is a list of documents loaded from the document_store folder.
    ''''

    if pd.isna(retrieved_doc_id) or str(retrieved_doc_id).strip() == "":
        return ""

    retrieved_doc_id = str(retrieved_doc_id).strip()

    # Search each loaded document and check whether the doc_id is present inside it.
    for document in document_store:
        content = str(document["content"])

        if f"doc_id: {retrieved_doc_id}" in content:
            return content

    return ""

def context_precision_check(reference_answer, retrieved_context, threshold=0.60):
    ''''
    Compares expected answer meaning with retrieved document content.

    Input:
    - reference_answer: gold_answer / expected answer
    - retrieved_context: retrieved document content

    Output:
    - context precision score
    - Pass/Fail status
    - remarks
    ''''

    if pd.isna(reference_answer) or str(reference_answer).strip() == "":
        return "", "Not Run", "Reference answer is missing."

    if pd.isna(retrieved_context) or str(retrieved_context).strip() == "":
        return "", "Not Run", "Retrieved document content is missing."

    # Convert expected answer into embedding.
    reference_embedding = embedding_model.encode(
        str(reference_answer),
        convert_to_tensor=True
    )

    # Convert retrieved document content into embedding.
    context_embedding = embedding_model.encode(
        str(retrieved_context),
        convert_to_tensor=True
    )

    # Compare meaning similarity between expected answer and retrieved document content.
    score = float(util.cos_sim(reference_embedding, context_embedding)[0][0])
    score = round(score, 4)

    if score >= threshold:
        return score, "Pass", "Retrieved document content is useful for answering the question."

    return score, "Fail", "Retrieved document content is not useful enough for answering the question."


# Create columns for storing retrieved document content and Context Precision result.
qa_df["actual_retrieved_doc_content"] = ""
qa_df["context_precision_score"] = ""
qa_df["context_precision_status"] = ""
qa_df["context_precision_remarks"] = ""


# Run Context Precision check for every row.
for row_index, row in qa_df.iterrows():

    # Step 1: Get the retrieved document content using actual_retrieved_doc_ids.

    retrieved_context = get_retrieved_document_content(
        retrieved_doc_id=row["actual_retrieved_doc_ids"],
        document_store=document_store
    )

    # Step 2: Store retrieved document content for QA traceability.
    qa_df.loc[row_index, "actual_retrieved_doc_content"] = retrieved_context

    # Step 3: Compare gold_answer with retrieved document content.
    score, status, remarks = context_precision_check(
        reference_answer=row["gold_answer"],
        retrieved_context=retrieved_context
    )

    # Step 4: Store Context Precision result.
    qa_df.loc[row_index, "context_precision_score"] = score
    qa_df.loc[row_index, "context_precision_status"] = status
    qa_df.loc[row_index, "context_precision_remarks"] = remarks


# Show Context Precision result.
display(
    qa_df[
        [
            "test_id",
            "gold_answer",
            "actual_retrieved_doc_ids",
            "context_precision_score",
            "context_precision_status",
            "context_precision_remarks"
        ]
    ]
)

"""

'\n\n## 9. Context Precision Check\n\n# This section checks whether the retrieved document/context is useful\n# for answering the question.\n#\n# Simple meaning:\n# - Retrieval check only checks whether the expected document ID was retrieved.\n# - Context Precision checks whether the retrieved document content is useful.\n#\n# Correct comparison:\n# gold_answer  vs  retrieved document content\n\n\ndef get_retrieved_document_content(retrieved_doc_id, document_store):\n    \'\'\'\'\n    Finds the retrieved document content using actual_retrieved_doc_ids.\n\n    Here document_store is a list of documents loaded from the document_store folder.\n    \'\'\'\'\n\n    if pd.isna(retrieved_doc_id) or str(retrieved_doc_id).strip() == "":\n        return ""\n\n    retrieved_doc_id = str(retrieved_doc_id).strip()\n\n    # Search each loaded document and check whether the doc_id is present inside it.\n    for document in document_store:\n        content = str(document["content"])\n\n        if f"do

In [31]:
# Groq model setup for DeepEval using custom wrapper
# This lets DeepEval use Groq instead of OpenAI.
# This also avoids Groq tool-call wrapper format issues.

!pip install deepeval groq -q

import os
import re
from google.colab import userdata
from groq import Groq
from deepeval.models import DeepEvalBaseLLM


os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")


class GroqDeepEvalModel(DeepEvalBaseLLM):

    #def __init__(self, model_name="llama-3.3-70b-versatile"):
    def __init__(self, model_name="llama-3.1-8b-instant"):
        self.model_name = model_name
        self.client = Groq(api_key=os.environ["GROQ_API_KEY"])

    def load_model(self):
        return self.client

    def clean_response(self, text):
        text = str(text).strip()

        # Remove Groq tool-call wrapper if Groq returns it.
        text = text.replace("<function=json_tool_call>", "")
        text = text.replace("</function>", "")

        # Keep only JSON content if extra text is present.
        json_match = re.search(r"(\{.*\}|\[.*\])", text, re.DOTALL)

        if json_match:
            return json_match.group(1).strip()

        return text

    def generate(self, prompt: str, schema=None, **kwargs):
        try:
            response = self.client.chat.completions.create(
                model=self.model_name,
                messages=[
                    {
                        "role": "system",
                        "content": "Return only valid JSON. Do not use tool calls. Do not wrap JSON in function tags. Escape all double quotes inside string values using backslash. Do not copy raw quoted text into JSON strings."
                    },
                    {
                        "role": "user",
                        "content": prompt
                    }
                ],
                temperature=0,
                response_format={"type": "json_object"},
                max_completion_tokens=2048
            )

        except Exception as error:
            raise Exception(f"Groq evaluator error: {str(error)}")


        cleaned_text = self.clean_response(response.choices[0].message.content)

        if schema is not None:
            return schema.model_validate_json(cleaned_text)

        return cleaned_text

    async def a_generate(self, prompt: str, **kwargs) -> str:
        return self.generate(prompt, **kwargs)

    def get_model_name(self):
        return self.model_name


groq_model = GroqDeepEvalModel()

print("Groq custom model connected for DeepEval.")


Groq custom model connected for DeepEval.


### How Tracing Supports Contextual Precision

```text
Retriever finds top-k similar documents.
Tracing records those retrieved documents.
Contextual Precision uses those traced documents for evaluation.

!rm -rf /content/document_store
!rm -rf /content/.deepeval
!rm -f /content/lightweight_rag_qa_dataset*.csv
!rm -f /content/rag_qa_dataset_with_demo_rag_outputs.csv

In [ ]:
## 9. Contextual Precision and Answer Relevancy Check using DeepEval Trace

# This section uses DeepEval's professional RAG metric with tracing.
#
# Contextual Precision checks whether useful retrieved contexts
# are ranked higher than less useful/noisy contexts.
#
# In this traced version:
# - Step 32 traced function stores retrieval_context inside DeepEval trace.
# - This cell provides input and expected_output using Golden.
# - DeepEval reads retrieval_context from the trace during evaluation.

!pip install deepeval -q

from deepeval.evaluate import ErrorConfig, AsyncConfig
import json
import os
from deepeval.dataset import Golden, EvaluationDataset
from deepeval.metrics import ContextualPrecisionMetric, AnswerRelevancyMetric


# Create result columns for Contextual Precision.
qa_df["contextual_precision_score"] = ""
qa_df["contextual_precision_status"] = ""
qa_df["contextual_precision_reason"] = ""


qa_df["answer_relevancy_score"] = ""
qa_df["answer_relevancy_status"] = ""
qa_df["answer_relevancy_reason"] = ""

# Create DeepEval Contextual Precision metric.
# Groq is used as the evaluation judge through our custom wrapper.
contextual_precision_metric = ContextualPrecisionMetric(
    threshold=0.6,
    model=groq_model,
    include_reason=True
)


answer_relevancy_metric = AnswerRelevancyMetric(
    threshold=0.6,
    model=groq_model,
    include_reason=True
)

# Create Golden test cases from qa_df.
# Each Golden contains:
# - input: original question
# - expected_output: gold_answer / expected answer
goldens = []

for row_index, row in qa_df.iterrows():
    goldens.append(
        Golden(
            input=row["question"],
            expected_output=row["gold_answer"]
        )
    )


# Create DeepEval evaluation dataset.
# This lets DeepEval run each Golden test case one by one.
evaluation_dataset = EvaluationDataset(
    goldens=goldens
)


# Run Contextual Precision using traced RAG function.
# During each run:
# - evals_iterator provides the current Golden
# - run_traced_rag_pipeline() retrieves top-k docs and stores them in trace
# - ContextualPrecisionMetric reads retrieval_context from the trace
import time

#for row_index, golden in enumerate(
#    evaluation_dataset.evals_iterator(metrics=[contextual_precision_metric])
#):

# Store only error reasons when errors happen.
#error_reason_file = "rag_metric_error_reasons.json"
error_reason_file = "/content/rag_metric_error_reasons.json"
error_reasons = {}

for row_index, golden in enumerate(
    evaluation_dataset.evals_iterator(
        metrics=[
            contextual_precision_metric,
            answer_relevancy_metric
        ],
        error_config=ErrorConfig(ignore_errors=True),
        async_config=AsyncConfig(run_async=False)
    )
):



    try:
        # Run traced RAG pipeline for this test case.
        # This call creates actual_output and retrieval_context inside DeepEval trace.
        generated_answer = run_traced_rag_pipeline(
            question=golden.input,
            expected_output=golden.expected_output,
            top_k=2
        )

        # Store actual output in CSV/report.
        qa_df.loc[row_index, "actual_output"] = generated_answer

        """
        # Store Contextual Precision result in CSV/report.
        qa_df.loc[row_index, "contextual_precision_score"] = contextual_precision_metric.score
        qa_df.loc[row_index, "contextual_precision_reason"] = contextual_precision_metric.reason

        if contextual_precision_metric.score >= 0.7:
            qa_df.loc[row_index, "contextual_precision_status"] = "Pass"
        else:
            qa_df.loc[row_index, "contextual_precision_status"] = "Fail"

        print("Completed:", qa_df.loc[row_index, "test_id"])
        """
                # In tracing style, DeepEval prints the metric result through evals_iterator.
        # So we do not manually compare contextual_precision_metric.score here.
        #qa_df.loc[row_index, "contextual_precision_status"] = "Completed"
        #qa_df.loc[row_index, "contextual_precision_reason"] = "Contextual Precision executed through DeepEval trace."

        # Store Contextual Precision result directly from metric object.
        precision_score = contextual_precision_metric.score
        precision_reason = contextual_precision_metric.reason

        qa_df.loc[row_index, "contextual_precision_score"] = precision_score
        qa_df.loc[row_index, "contextual_precision_reason"] = precision_reason

        if precision_score is None:
            qa_df.loc[row_index, "contextual_precision_status"] = "Error"
        elif precision_score >= 0.6:
            qa_df.loc[row_index, "contextual_precision_status"] = "Pass"
        else:
            qa_df.loc[row_index, "contextual_precision_status"] = "Fail"




        # Store Answer Relevancy result directly from metric object.
        answer_relevancy_score = answer_relevancy_metric.score
        answer_relevancy_reason = answer_relevancy_metric.reason

        qa_df.loc[row_index, "answer_relevancy_score"] = answer_relevancy_score
        qa_df.loc[row_index, "answer_relevancy_reason"] = answer_relevancy_reason

        if answer_relevancy_score is None:
            qa_df.loc[row_index, "answer_relevancy_status"] = "Error"
        elif answer_relevancy_score >= 0.6:
            qa_df.loc[row_index, "answer_relevancy_status"] = "Pass"
        else:
            qa_df.loc[row_index, "answer_relevancy_status"] = "Fail"



    except Exception as error:
        error_text = str(error)

        error_reasons[qa_df.loc[row_index, "test_id"]] = {
            "contextual_precision_reason": error_text,
            "answer_relevancy_reason": error_text
        }

        with open(error_reason_file, "w", encoding="utf-8") as file:
            json.dump(error_reasons, file, indent=4)

        qa_df.loc[row_index, "contextual_precision_status"] = "Error"
        qa_df.loc[row_index, "contextual_precision_reason"] = error_text


        qa_df.loc[row_index, "answer_relevancy_status"] = "Error"
        qa_df.loc[row_index, "answer_relevancy_reason"] = error_text

        print("Error:", qa_df.loc[row_index, "test_id"], error_text)



    # Small delay to reduce free-tier rate-limit issues.
    time.sleep(60)



# Show Contextual Precision result.
display(
    qa_df[
        [
            "test_id",
            "question",
            "gold_answer",
            "contextual_precision_score",
            "contextual_precision_status",
            "contextual_precision_reason",
            "answer_relevancy_score",
            "answer_relevancy_status",
            "answer_relevancy_reason"
        ]
    ]
)

Output()

In [ ]:
## 10. Contextual Recall Check using DeepEval Trace

# This section runs only Contextual Recall.
#
# Contextual Recall checks whether the retrieved top-k documents
# contain enough information from the expected/gold answer.
#
# We keep this separate because Contextual Recall creates larger JSON output
# and can fail more easily when combined with other metrics.

from deepeval.evaluate import ErrorConfig, AsyncConfig
from deepeval.dataset import Golden, EvaluationDataset
from deepeval.metrics import ContextualRecallMetric

import json
import time


# Create result columns for Contextual Recall.
qa_df["contextual_recall_score"] = ""
qa_df["contextual_recall_status"] = ""
qa_df["contextual_recall_reason"] = ""


# Create DeepEval Contextual Recall metric.
# Groq is used as the evaluation judge through our custom wrapper.
contextual_recall_metric = ContextualRecallMetric(
    threshold=0.6,
    model=groq_model,
    include_reason=True
)


# Create Golden test cases from qa_df.
# Each Golden contains:
# - input: original question
# - expected_output: gold_answer / expected answer
goldens = []

for row_index, row in qa_df.iterrows():
    goldens.append(
        Golden(
            input=row["question"],
            expected_output=row["gold_answer"]
        )
    )


# Create DeepEval evaluation dataset.
evaluation_dataset = EvaluationDataset(
    goldens=goldens
)


# Store only Contextual Recall error reasons when errors happen.
contextual_recall_error_reason_file = "/content/contextual_recall_error_reasons.json"
contextual_recall_error_reasons = {}


for row_index, golden in enumerate(
    evaluation_dataset.evals_iterator(
        metrics=[
            contextual_recall_metric
        ],
        error_config=ErrorConfig(ignore_errors=True),
        async_config=AsyncConfig(run_async=False)
    )
):

    try:
        # Run traced RAG pipeline for this test case.
        # This creates retrieval_context inside DeepEval trace.
        run_traced_rag_pipeline(
            question=golden.input,
            expected_output=golden.expected_output,
            top_k=2
        )

        # Store Contextual Recall result directly from metric object.
        recall_score = contextual_recall_metric.score
        recall_reason = contextual_recall_metric.reason

        qa_df.loc[row_index, "contextual_recall_score"] = recall_score
        qa_df.loc[row_index, "contextual_recall_reason"] = recall_reason

        if recall_score is None:
            qa_df.loc[row_index, "contextual_recall_status"] = "Error"
        elif recall_score >= 0.6:
            qa_df.loc[row_index, "contextual_recall_status"] = "Pass"
        else:
            qa_df.loc[row_index, "contextual_recall_status"] = "Fail"

    except Exception as error:
        error_text = str(error)

        contextual_recall_error_reasons[qa_df.loc[row_index, "test_id"]] = {
            "contextual_recall_reason": error_text
        }

        with open(contextual_recall_error_reason_file, "w", encoding="utf-8") as file:
            json.dump(contextual_recall_error_reasons, file, indent=4)

        qa_df.loc[row_index, "contextual_recall_status"] = "Error"
        qa_df.loc[row_index, "contextual_recall_reason"] = error_text

        print("Contextual Recall Error:", qa_df.loc[row_index, "test_id"], error_text)

    # Small delay to reduce free-tier rate-limit issues.
    time.sleep(60)


# Show Contextual Recall result.
display(
    qa_df[
        [
            "test_id",
            "question",
            "gold_answer",
            "contextual_recall_score",
            "contextual_recall_status",
            "contextual_recall_reason"
        ]
    ]
)

In [ ]:
## 10A. Fill Missing Error Reasons

# This step runs only for rows where:
# - metric status is Error
# - metric reason is empty
#
# Purpose:
# Normal DeepEval tracing flow is kept unchanged.
# This retry is only for capturing the real error reason for reporting.

import pandas as pd
from deepeval.test_case import LLMTestCase


def get_top_k_contexts_for_error_check(question, top_k=2):
    """
    Retrieves top-k contexts again only for error-reason debugging.
    This does not change the main RAG evaluation flow.
    """

    best_document, top_k_results = vector_retrieve(
        question=question,
        top_k=top_k
    )

    top_k_contexts = [
        document["content"]
        for document in top_k_results
    ]

    return top_k_contexts


def fill_missing_metric_error_reason(row_index, metric_name, metric_object):
    """
    Runs one failed metric directly to capture the real exception reason.
    This is used only when status is Error and reason is empty.
    """

    row = qa_df.loc[row_index]

    status_column = f"{metric_name}_status"
    reason_column = f"{metric_name}_reason"

    status = str(row[status_column]).strip()

    reason_value = row[reason_column]

    if pd.isna(reason_value):
        reason = ""
    else:
        reason = str(reason_value).strip()

    if status != "Error":
        return

    if reason != "":
        return

    try:
        retrieval_context = get_top_k_contexts_for_error_check(
            question=row["question"],
            top_k=2
        )

        test_case = LLMTestCase(
            input=row["question"],
            actual_output=row["actual_output"],
            expected_output=row["gold_answer"],
            retrieval_context=retrieval_context
        )

        metric_object.measure(test_case)

        if metric_object.reason is not None and str(metric_object.reason).strip() != "":
            qa_df.loc[row_index, reason_column] = metric_object.reason
        else:
            qa_df.loc[row_index, reason_column] = "Metric returned Error but did not provide a reason."

    except Exception as error:
        qa_df.loc[row_index, reason_column] = str(error)


for row_index, row in qa_df.iterrows():

    fill_missing_metric_error_reason(
        row_index=row_index,
        metric_name="contextual_precision",
        metric_object=contextual_precision_metric
    )

    fill_missing_metric_error_reason(
        row_index=row_index,
        metric_name="contextual_recall",
        metric_object=contextual_recall_metric
    )

    fill_missing_metric_error_reason(
        row_index=row_index,
        metric_name="answer_relevancy",
        metric_object=answer_relevancy_metric
    )


display(
    qa_df[
        [
            "test_id",
            "contextual_precision_status",
            "contextual_precision_reason",
            "contextual_recall_status",
            "contextual_recall_reason",
            "answer_relevancy_status",
            "answer_relevancy_reason"
        ]
    ]
)

In [ ]:
# Save and download Contextual Precision tracing result

# This CSV is for the current completed stage:
# DeepEval Contextual Precision with tracing.

#contextual_precision_output_file = "rag_contextual_precision_tracing_result.csv"
rag_metrics_output_file = "rag_contextual_precision_recall_tracing_result.csv"


#qa_df.to_csv(contextual_precision_output_file, index=False)

contextual_precision_result_df = qa_df[
    [
        "test_id",
        "question",
        "expected_doc_ids",
        "gold_answer",
        "actual_output",
        "contextual_precision_status",
        "contextual_precision_score",
        "contextual_precision_reason",
        "contextual_recall_score",
        "contextual_recall_status",
        "contextual_recall_reason",
        "answer_relevancy_score",
        "answer_relevancy_status",
        "answer_relevancy_reason"
    ]
]

#contextual_precision_result_df.to_csv(contextual_precision_output_file, index=False)

#print("Saved:", contextual_precision_output_file)


from google.colab import files

#files.download(contextual_precision_output_file)


contextual_precision_result_df.to_csv(rag_metrics_output_file, index=False)

print("Saved:", rag_metrics_output_file)

files.download(rag_metrics_output_file)